In [ ]:
import cv2
import numpy as np

sift = cv2.SIFT_create()

def count_keypoints(imgs):
    counts = []
    for img in imgs:
        if img.dtype != np.uint8:
            img = img.astype(np.uint8)
        kp, desc = sift.detectAndCompute(img, None)
        counts.append(0 if kp is None else len(kp))
    return np.array(counts)

counts = count_keypoints(train_imgs)  # train_imgs from your Dataset

print("num images:", len(counts))
print("min keypoints:", counts.min())
print("max keypoints:", counts.max())
print("mean keypoints:", counts.mean())
print("median keypoints:", np.median(counts))
print("total keypoints:", counts.sum())


In [5]:
from sklearn.preprocessing import LabelEncoder
from dataset import Dataset
from siftbovw import SiftBoVW
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

PARAM = {
    'n_words': 100,
    'max_sample': 100000,
    'batch_size': 100,
    'random_state': 0
}

N_NEIGHBORS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
# le = LabelEncoder()
training = Dataset(r"training",img_size=(256, 256))
training.shuffle_data()
imgs, labels = training.get_data()
split_ratio = 0.9
split_index = int(len(imgs) * split_ratio)
imgs_train, labels_train = imgs[:split_index], labels[:split_index]
imgs_test, labels_test = imgs[split_index:], labels[split_index:]


print("Parameters:", PARAM)

sift_bovw = SiftBoVW(n_words=PARAM['n_words'], max_sample=PARAM['max_sample'], batch_size=PARAM['batch_size'], random_state=PARAM['random_state'])
x_train = sift_bovw.fit_transform(imgs_train)

# Check the cluster centers
center = sift_bovw.kmeans.cluster_centers_

x_test = sift_bovw.transform(imgs_test)
for N_NEIGHBOR in N_NEIGHBORS:
    print("Number of neighbors:", N_NEIGHBOR)
    knn = KNeighborsClassifier(n_neighbors= N_NEIGHBOR)
    knn.fit(x_train, labels_train)
    y_pred = knn.predict(x_test)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"Accuracy: {accuracy*100:.2f}%")


Parameters: {'n_words': 100, 'max_sample': 100000, 'batch_size': 100, 'random_state': 0}
[SIFTBoVW] Total descriptors before sampling: 696266
[SIFTBoVW] Subsampled to 100000 descriptors
Number of neighbors: 10
Accuracy: 40.67%
Number of neighbors: 20
Accuracy: 38.67%
Number of neighbors: 30
Accuracy: 44.67%
Number of neighbors: 40
Accuracy: 42.00%
Number of neighbors: 50
Accuracy: 39.33%
Number of neighbors: 60
Accuracy: 42.67%
Number of neighbors: 70
Accuracy: 42.00%
Number of neighbors: 80
Accuracy: 39.33%
Number of neighbors: 90
Accuracy: 39.33%
Number of neighbors: 100
Accuracy: 39.33%
